In [2]:
from pathlib import Path

LAST_NAME = "Fuentes"
STUDENT_ID = "TUPM-26-1908"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Seed: 8
ID Sum: 26
Name Length: 7


In [3]:
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Student: Fuentes
Student ID: TUPM-26-1908
Workspace: C:\Users\Andee Fuentes\OneDrive\Desktop\New folder\ffmpeg-snapshot\ffmpeg\libavfilter
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [4]:
import urllib.request

datasets = {
    "weblog.csv": "https://raw.githubusercontent.com/shawon100/Web-Log-Dataset/master/weblog.csv",
    "user_behavior_dataset.csv": "https://raw.githubusercontent.com/RonBalaban/CUNY-SPS/refs/heads/main/DATA622/user_behavior_dataset.csv",
    "Global_Cybersecurity_Threats_2015-2024.csv": "https://raw.githubusercontent.com/InimKelvin/Cybersecurity-Threat-Analysis/main/Global_Cybersecurity_Threats_2015-2024.csv"
}
for filename, url in datasets.items():
    urllib.request.urlretrieve(url, DATA_DIR / filename)
    print("Downloaded:", filename)

Downloaded: weblog.csv
Downloaded: user_behavior_dataset.csv
Downloaded: Global_Cybersecurity_Threats_2015-2024.csv


In [5]:
# Web Server Access Log Analysis

# [EX1] Web log analyzer (line-by-line, no full load)
import json
import shutil

records_to_process = 1000 + ID_SUM          # 1026
target_class = (SEED_DIGIT % 4) + 2         # 2 -> 2xx
class_digit = str(target_class)

log_path = DATA_DIR / "weblog.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"

processed = 0
in_class = 0
status_counts = {}
url_counts = {}

with open(log_path, "r", encoding="utf-8") as source:
    header_line = source.readline().strip()
    with open(filtered_path, "w", encoding="utf-8") as output:
        output.write(header_line + "\n")
        for _ in range(records_to_process):
            line = source.readline()
            if not line:
                break
            fields = line.strip().split(",")
            if len(fields) > 3 and fields[3].isdigit():   # skip invalid/missing
                ip, time, url, status = fields[0], fields[1], fields[2], fields[3]
                processed += 1
                if status[0] == class_digit:              # target status class
                    in_class += 1
                    if status in status_counts:
                        status_counts[status] += 1
                    else:
                        status_counts[status] = 1
                    if url in url_counts:
                        url_counts[url] += 1
                    else:
                        url_counts[url] = 1
                    output.write(line.strip() + "\n")

print("Filtered CSV file:", filtered_path)
print("Records processed:", processed)
print("Target status class:", f"{target_class}xx")
print("Records in target class:", in_class)
print("Count per status code:", status_counts)

# five most requested URLs (pick the max 5 times; basic loop only)
top_urls = []
top_counts = []
for _ in range(5):
    best_url = ""
    best_count = 0
    for url in url_counts:
        if url_counts[url] > best_count and url not in top_urls:
            best_count = url_counts[url]
            best_url = url
    if best_url != "":
        top_urls.append(best_url)
        top_counts.append(best_count)

print("Five most requested URLs:")
for url, count in zip(top_urls, top_counts):
    print(f"{url}: {count}")

summary = {
    "records_processed": processed,
    "target_status_class": f"{target_class}xx",
    "records_in_target_class": in_class,
    "count_per_status_code": status_counts,
    "five_most_requested_urls": top_urls,
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
print("JSON summary file:", json_path)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)
print("Backup verification:", backup_path.exists())

Filtered CSV file: output\Fuentes_TUPM-26-1908_weblog_filtered.csv
Records processed: 1011
Target status class: 2xx
Records in target class: 842
Count per status code: {'200': 834, '206': 8}
Five most requested URLs:
GET /login.php HTTP/1.1: 103
GET /home.php HTTP/1.1: 91
GET /css/normalize.css HTTP/1.1: 38
GET /css/bootstrap.min.css HTTP/1.1: 37
GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1: 37
JSON summary file: output\Fuentes_TUPM-26-1908_weblog_summary.json
Backup verification: True


In [6]:
# Mobile Device Usage Profiling

# [EX2] Mobile usage profiler
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"   # Android
target_behavior = (SEED_DIGIT % 5) + 1                    # 4

behavior_path = DATA_DIR / "user_behavior_dataset.csv"
export_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_selected.csv"

total_records = 0
os_records = 0
selected = 0
total_drain = 0.0
total_usage = 0.0

with open(behavior_path, "r", encoding="utf-8") as source:
    header_line = source.readline().strip()
    with open(export_path, "w", encoding="utf-8") as output:
        output.write("User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage (MB/day)\n")
        for line in source:
            total_records += 1
            fields = line.strip().split(",")
            if len(fields) > 6 and fields[2] == target_os:
                os_records += 1
                if fields[6].isdigit() and int(fields[6]) == target_behavior:
                    selected += 1
                    total_drain += float(fields[3])
                    total_usage += float(fields[4])
                    output.write(f"{fields[0]},{fields[1]},{fields[2]},{fields[3]},{fields[4]}\n")

avg_drain = round(total_drain / selected, 3) if selected > 0 else 0
avg_usage = round(total_usage / selected, 3) if selected > 0 else 0

with open(export_path, "r", encoding="utf-8") as file:
    next(file)
    exported_rows = len(file.readlines())

print("Operating system and behavior class:", target_os, target_behavior)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_records)
print("Records in target behavior class:", selected)
print("Average battery drain (mAh/day):", avg_drain)
print("Average data usage (MB/day):", avg_usage)
print("Count verification:", exported_rows == selected)
print("Exported CSV file:", export_path)

summary = {
    "operating_system": target_os,
    "behavior_class": target_behavior,
    "total_records": total_records,
    "os_matching_records": os_records,
    "selected_records": selected,
    "average_battery_drain": avg_drain,
    "average_data_usage": avg_usage,
    "count_verification": exported_rows == selected,
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
print("JSON summary file:", json_path)

backup_path = BACKUP_DIR / export_path.name
shutil.copy2(export_path, backup_path)
print("Backup file:", backup_path)
print("Backup file size:", backup_path.stat().st_size, "bytes")

Operating system and behavior class: Android 4
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 0
Average battery drain (mAh/day): 0
Average data usage (MB/day): 0
Count verification: True
Exported CSV file: output\Fuentes_TUPM-26-1908_mobile_selected.csv
JSON summary file: output\Fuentes_TUPM-26-1908_mobile_summary.json
Backup file: backup\Fuentes_TUPM-26-1908_mobile_selected.csv
Backup file size: 83 bytes


In [7]:
#Cyber Security Incident Report

# [EX3] Cybersecurity incident reporting program
target_year = 2015 + SEED_DIGIT              # 2023

threats_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"

total_records = 0
incidents = 0
loss_records = 0
total_loss = 0.0
attack_counts = {}
year_records = []                            # (loss, date, attack) in original order

with open(threats_path, "r", encoding="utf-8") as source:
    header_line = source.readline().strip()
    header = header_line.split(",")
    date_idx = -1
    attack_idx = -1
    loss_idx = -1
    for i, name in enumerate(header):        # locate columns like Cell 6 enumerate
        if "Date" in name:
            date_idx = i
        if "Attack" in name:
            attack_idx = i
        if "Financial Loss" in name:
            loss_idx = i
    with open(filtered_path, "w", encoding="utf-8") as output:
        output.write(header_line + "\n")
        for line in source:
            total_records += 1
            fields = line.strip().split(",")
            if len(fields) > loss_idx and str(target_year) in fields[date_idx]:
                incidents += 1
                output.write(line.strip() + "\n")
                attack = fields[attack_idx]
                if attack in attack_counts:
                    attack_counts[attack] += 1
                else:
                    attack_counts[attack] = 1
                value = fields[loss_idx]
                if value.replace(".", "", 1).isdigit():   # handle invalid loss values
                    loss = float(value)
                    total_loss += loss
                    loss_records += 1
                    year_records.append((loss, fields[date_idx], attack))

average_loss = round(total_loss / loss_records, 3) if loss_records > 0 else 0

# five costliest: pick max 5 times; strict > keeps original order on ties
costliest = []
used = []
for _ in range(5):
    best_i = -1
    best_loss = -1.0
    for i in range(len(year_records)):
        if i not in used and year_records[i][0] > best_loss:
            best_loss = year_records[i][0]
            best_i = i
    if best_i >= 0:
        used.append(best_i)
        costliest.append(year_records[best_i])

print("Target year:", target_year)
print("Total records in dataset:", total_records)
print("Incidents in target year:", incidents)
print("Count per attack type:", attack_counts)
print("Total financial loss (USD millions):", round(total_loss, 3))
print("Average financial loss (USD millions):", average_loss)
print("Five costliest incidents:")
for loss, date, attack in costliest:
    print(f"{date} | {attack} | {loss:.3f}")
print("Filtered CSV file:", filtered_path)

summary = {
    "target_year": target_year,
    "total_records": total_records,
    "incidents_in_target_year": incidents,
    "count_per_attack_type": attack_counts,
    "total_financial_loss": round(total_loss, 3),
    "average_financial_loss": average_loss,
    "five_costliest": [f"{d} | {a} | {l:.3f}" for l, d, a in costliest]
}
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_summary.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
print("JSON summary file:", json_path)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)
print("Backup verification:", backup_path.exists())

report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Incidents in Target Year: {incidents}\n")
    report.write(f"Total Financial Loss: {total_loss:.3f}\n")
    report.write(f"Average Financial Loss: {average_loss:.3f}\n")
    report.write("Count per Attack Type:\n")
    for attack in attack_counts:
        report.write(f"  {attack}: {attack_counts[attack]}\n")
    report.write("Five Costliest Incidents:\n")
    for loss, date, attack in costliest:
        report.write(f"  {date} | {attack} | {loss:.3f}\n")
    report.write(f"Filtered CSV File: {filtered_path.name}\n")
    report.write(f"JSON Summary File: {json_path.name}\n")
    report.write(f"Backup File: {backup_path.name}\n")
print("Report file:", report_path)

Target year: 2023
Total records in dataset: 3000
Incidents in target year: 0
Count per attack type: {}
Total financial loss (USD millions): 0.0
Average financial loss (USD millions): 0
Five costliest incidents:
Filtered CSV file: output\Fuentes_TUPM-26-1908_cyber_filtered.csv
JSON summary file: output\Fuentes_TUPM-26-1908_cyber_summary.json
Backup verification: True
Report file: output\Fuentes_TUPM-26-1908_cyber_report.txt
